# Phase 5 — Applied FYF Model and Scenarios

Companion notebook to `notes/phase5-fyf-model.md`. We:

1. Walk Q1 month by month for the on-target scenario, reproducing
   exercise 1's expected values.
2. Compute the Q1 FYF year-end forecast and P(over-budget).
3. Demonstrate the surprise diagnostic on the budget-shock scenario.
4. Run all five canonical scenarios and tabulate quarterly summaries.
5. Run the calibration check from exercise 5.

In [ ]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

from src.conjugate import NormalPosterior
from src.fyf_model import (
    FYFConfig, FYFModel,
    build_scenarios, run_scenario, run_scenario_secondary,
)
from src.diagnostics import (
    surprise_score,
    calibration_score, calibration_binomial_test,
    priors_before_each_step,
)
from src.visualization import (
    plot_posterior_evolution, plot_surprise_scores,
    plot_year_end_forecast, plot_p_over_budget,
    plot_fyf_comparison,
)

rng = np.random.default_rng(seed=20260901)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True})

## 1. Q1 walkthrough (Exercise 1)

Default config; Q1 actuals from `docs/model-design.md` §5.

In [ ]:
config = FYFConfig(
    prior_mean=1_050_000.0,
    prior_sd=150_000.0,
    obs_sd=80_000.0,
    budget_ceiling=13_200_000.0,
)
model = FYFModel(config)

q1_actuals = [1_120_000.0, 1_095_000.0, 1_080_000.0]
for x in q1_actuals:
    review = model.process_month(x)
    s = review.posterior
    print(f"month {review.month:>2}: x = R$ {review.actual:>10,.0f} | μ_n = R$ {s.mean():>10,.0f} | σ_n = R$ {s.std():>9,.0f} | z = {review.surprise_z:+.3f}")

## 2. Q1 FYF year-end forecast (Exercise 2)

P(annual total > 13.2M)?

In [ ]:
q1 = model.fyf_review(1)
print(f"Q1 FYF Review")
print(f"  posterior:        N({q1.posterior.mean():,.0f}, {q1.posterior.std():,.0f}^2)")
if q1.forecast is not None:
    print(f"  forecast E[T]:    R$ {q1.forecast.mean:,.0f}")
    print(f"  forecast σ_T:     R$ {q1.forecast.std():,.0f}")
print(f"  P(T > B):         {q1.p_over_budget:.4f}")
print(f"  max |z|:          {q1.max_abs_surprise:.3f}")
print(f"  recommendation:   {q1.recommendation}")

## 3. Run all 5 scenarios and tabulate

Compact summary across the canonical Phase-5 scenarios.

In [ ]:
rows = []
for scenario in build_scenarios():
    if scenario.id == "s5_prior_sensitivity":
        m_a = run_scenario(scenario)
        m_b = run_scenario_secondary(scenario)
        for label, m in [("S5 confident", m_a), ("S5 uncertain", m_b)]:
            qr = m.fyf_review(4)
            rows.append((label, qr.posterior.mean(), qr.posterior.std(), qr.max_abs_surprise, qr.recommendation))
    else:
        m = run_scenario(scenario)
        qr = m.fyf_review(4)
        rows.append((scenario.name, qr.posterior.mean(), qr.posterior.std(), qr.max_abs_surprise, qr.recommendation))

print(f"{'Scenario':25} {'μ_12':>14} {'σ_12':>10} {'max|z|':>8} {'recommendation':>22}")
print("-" * 85)
for r in rows:
    print(f"{r[0]:25} R$ {r[1]:>10,.0f} R$ {r[2]:>7,.0f} {r[3]:>8.3f} {r[4]:>22}")

## 4. Visualise the budget-shock scenario

The 4-panel figure produced by `scripts/fig_fyf_scenarios.py`,
reproduced here for S3 to highlight the shock detection.

In [ ]:
scenarios = {s.id: s for s in build_scenarios()}
shock = scenarios["s3_shock"]
model_shock = run_scenario(shock)

fig, axes = plt.subplots(2, 2, figsize=(13, 8.4), constrained_layout=True)
plot_posterior_evolution(axes[0, 0], model_shock.reviews(), prior_mean=shock.config.prior_mean,
                         title="Posterior trajectory — shock")
plot_surprise_scores(axes[0, 1], model_shock.reviews(), title="Surprise z-scores — shock")
plot_year_end_forecast(axes[1, 0], model_shock.reviews(),
                       budget_ceiling=shock.config.budget_ceiling,
                       title="Year-end total forecast")
plot_p_over_budget(axes[1, 1], model_shock.reviews(), title="P(annual total > B)")
fig.suptitle("S3 — Budget shock: month-5 anomaly visible in z-scores and forecast", fontsize=11)
plt.show()

# Verify the diagnostic fired
z5 = model_shock.reviews()[4].surprise_z
print(f"\nMonth 5 surprise z = {z5:+.2f}  → diagnostic fires (|z|>3)" if abs(z5) > 3 else f"\nMonth 5 surprise z = {z5:+.2f}")

## 5. Prior sensitivity (S5)

In [ ]:
ps = scenarios["s5_prior_sensitivity"]
m_conf = run_scenario(ps)
m_unc = run_scenario_secondary(ps)

fig, ax = plt.subplots(figsize=(8, 4.5))
plot_fyf_comparison(
    ax, [m_conf, m_unc],
    labels=[fr"Confident σ₀={ps.config.prior_sd:,.0f}",
            fr"Uncertain σ₀={ps.secondary_config.prior_sd:,.0f}"],
    title="Two priors converge under the same data",
)
ax.axhline(ps.config.prior_mean, color="black", ls=":", alpha=0.4)
fig.tight_layout(); plt.show()

# Final gap
diff_12 = abs(m_conf.reviews()[-1].posterior.mean() - m_unc.reviews()[-1].posterior.mean())
print(f"Year-end posterior mean gap: R$ {diff_12:,.0f}")

## 6. Calibration check (Exercise 5)

Simulate 5 years of monthly forecasts under correct specification
(60 trials), count how many fall inside the 95% predictive interval,
and run the binomial test.

In [ ]:
rng_calib = np.random.default_rng(seed=20260910)
all_priors = []
all_actuals = []
for year in range(5):
    config_y = FYFConfig(prior_mean=1_050_000, prior_sd=150_000, obs_sd=80_000)
    model_y = FYFModel(config_y)
    actuals = list(rng_calib.normal(1_080_000, 80_000, size=12))
    prior = NormalPosterior(mu=config_y.prior_mean, sigma_sq=config_y.prior_sd**2)
    priors_year = priors_before_each_step(
        # need to feed first to generate reviews
        [], prior
    )
    model_y.annual_cycle(actuals)
    priors_year = priors_before_each_step(model_y.reviews(), prior)
    all_priors.extend(priors_year)
    all_actuals.extend(actuals)

score = calibration_score(all_priors, all_actuals, sigma_sq=80_000**2, level=0.95)
test = calibration_binomial_test(all_priors, all_actuals, sigma_sq=80_000**2, level=0.95)
print(f"Total months:           {test.n_observed}")
print(f"Inside 95% interval:    {test.n_inside}")
print(f"Empirical coverage:     {test.empirical_coverage:.3f}")
print(f"Nominal coverage:       {test.nominal_coverage:.3f}")
print(f"Two-sided p-value:      {test.p_value_two_sided:.3f}")
print(f"Calibrated at α=0.05:   {test.is_calibrated(0.05)}")

---

**Next phase.** Phase 6 runs the full battery of experiments and
produces all publication-quality figures (300 DPI, fixed seeds), plus
an animated GIF of posterior evolution. Phase 7 writes the article.